In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

DATA_DIR = Path(r"../datasets")
PROCESSED_DIR = DATA_DIR / "processed"

train = pd.read_csv(PROCESSED_DIR / "ratings_train.csv")
test = pd.read_csv(PROCESSED_DIR / "ratings_test.csv")

print(train.shape, test.shape)


(19936012, 4) (5064083, 4)


In [2]:

## Create compact user and movie indexes

user_ids = train["userId"].unique()
movie_ids = train["movieId"].unique()

user_to_index = pd.Series(
    np.arange(len(user_ids)),
    index=user_ids
)

movie_to_index = pd.Series(
    np.arange(len(movie_ids)),
    index=movie_ids
)

rows = train["userId"].map(user_to_index).to_numpy()
cols = train["movieId"].map(movie_to_index).to_numpy()
values = train["rating"].to_numpy(dtype=np.float32)

user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)

print("User-item matrix:", user_item_matrix.shape)
print("Non-zero ratings:", user_item_matrix.nnz)


User-item matrix: (162541, 56559)
Non-zero ratings: 19936012


In [3]:
## Measure sparsity
density = (
    user_item_matrix.nnz /
    (user_item_matrix.shape[0] * user_item_matrix.shape[1])
)

print(f"Density: {density:.6%}")
print(f"Sparsity: {1-density:.6%}")


Density: 0.216857%
Sparsity: 99.783143%


In [4]:
## Matrix factorization

N_COMPONENTS = 50

svd = TruncatedSVD(
    n_components=N_COMPONENTS,
    random_state=42
)

user_factors = svd.fit_transform(user_item_matrix)
movie_factors = svd.components_.T

print("User factors:", user_factors.shape)
print("Movie factors:", movie_factors.shape)

print("Explained variance:",
      svd.explained_variance_ratio_.sum())


User factors: (162541, 50)
Movie factors: (56559, 50)
Explained variance: 0.2952823


In [ ]:
## Predict user/movie scores
def collaborative_scores(user_id):
    if user_id not in user_to_index.index:
        return None

    uidx = int(user_to_index[user_id])

    return user_factors[uidx] @ movie_factors.T


In [5]:
## Generate collaborative recommendations
def recommend_collaborative(user_id, n=10):
    scores = collaborative_scores(user_id)

    if scores is None:
        return pd.DataFrame()

    result = pd.DataFrame({
        "movieId": movie_ids,
        "collaborative_score": scores
    })

    rated_ids = set(
        train.loc[
            train["userId"] == user_id,
            "movieId"
        ]
    )

    result = result[
        ~result["movieId"].isin(rated_ids)
    ]

    return (
        result
        .sort_values(
            "collaborative_score",
            ascending=False
        )
        .head(n)
        .reset_index(drop=True)
    )


In [ ]:
test_user = int(train["userId"].iloc[0])

display(
    recommend_collaborative(test_user, 10)
)


In [6]:
## Save collaborative artifacts

from scipy.sparse import save_npz

np.save(
    PROCESSED_DIR / "cf_user_ids.npy",
    user_ids
)

np.save(
    PROCESSED_DIR / "cf_movie_ids.npy",
    movie_ids
)

np.save(
    PROCESSED_DIR / "cf_user_factors.npy",
    user_factors
)

np.save(
    PROCESSED_DIR / "cf_movie_factors.npy",
    movie_factors
)

print("Collaborative filtering artifacts saved.")


Collaborative filtering artifacts saved.


In [7]:
## Phase 5 validation
assert user_factors.shape[0] == len(user_ids)
assert movie_factors.shape[0] == len(movie_ids)

print("PHASE 5 COLLABORATIVE FILTERING VALIDATION PASSED")


PHASE 5 COLLABORATIVE FILTERING VALIDATION PASSED
